# 07 — Deep Learning: from one neuron to an MLP (PyTorch)

Phase 3. We build up deep learning from its atom:
1. A **single neuron** is logistic regression (weighted sum + sigmoid).
2. Training = one 5-step loop (forward -> loss -> gradient -> update -> repeat). **PyTorch autograd**
   computes the gradients for us.
3. Stack neurons into **hidden layers** with **ReLU** nonlinearities -> a multi-layer perceptron (MLP).
4. **Honest finale:** MLP vs XGBoost on static tabular data. XGBoost wins — which tells us *where*
   deep learning actually belongs (sequences, next phase).

> **Note:** this notebook imports **torch only, not xgboost**. On this Mac the two segfault when
> loaded in the same process (OpenMP conflict). XGBoost's score below is the reference number from
> notebook 04, measured on the identical data/split.

## 1. A single neuron in PyTorch = logistic regression

`nn.Linear(3,1)` is one neuron (3 weights + bias). `BCEWithLogitsLoss` = sigmoid + cross-entropy.
The training loop is the same 5 steps we did by hand; `loss.backward()` (autograd) replaces the
gradient math. It lands on the same weights scikit-learn finds.

In [1]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score
torch.manual_seed(42); np.random.seed(42)

df = pd.read_csv("../data/raw/cs-training.csv", index_col=0)
feats = ["RevolvingUtilizationOfUnsecuredLines","age","NumberOfTime30-59DaysPastDueNotWorse"]
X = df[feats].copy()
X["RevolvingUtilizationOfUnsecuredLines"] = X["RevolvingUtilizationOfUnsecuredLines"].clip(0,2)
X["age"] = X["age"].replace(0, X["age"].median())
X["NumberOfTime30-59DaysPastDueNotWorse"] = X["NumberOfTime30-59DaysPastDueNotWorse"].clip(0,10)
X = ((X - X.mean())/X.std()).values
y = df["SeriousDlqin2yrs"].values
Xt = torch.tensor(X, dtype=torch.float32)
yt = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

neuron = nn.Linear(3,1)
criterion = nn.BCEWithLogitsLoss()
opt = torch.optim.SGD(neuron.parameters(), lr=0.5)
for step in range(3000):
    opt.zero_grad()           # 1 clear grads
    z = neuron(Xt)            # 2 forward (weighted sum)
    loss = criterion(z, yt)   # 3 loss
    loss.backward()           # 4 autograd computes gradients
    opt.step()                # 5 update weights
with torch.no_grad():
    p = torch.sigmoid(neuron(Xt)).numpy().ravel()
print("weights", np.round(neuron.weight.detach().numpy().ravel(),3), "bias", round(neuron.bias.item(),3))
print("AUC", round(roc_auc_score(y,p),4), " (= exactly what sklearn LogisticRegression finds)")

weights [ 0.752 -0.25   0.411] bias -3.131
AUC 0.8213  (= exactly what sklearn LogisticRegression finds)


## 2. Hidden layers + ReLU = a real network

One neuron draws a straight line. Stacking `Linear` layers alone stays linear — you need a
**nonlinearity** between them. **ReLU** = `max(0, x)` provides the kink that lets the network learn
curves and interactions. An MLP is: `Linear -> ReLU -> Linear -> ReLU -> Linear`.

## 3. Train an MLP on the full feature set

Neural nets can't eat NaN or raw scales, so unlike XGBoost we must **impute** (fill gaps with the
train median) and **standardize** (mean 0, std 1). We fight the 6.7% imbalance with `pos_weight`.

In [2]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
torch.manual_seed(42); np.random.seed(42)

TARGET="SeriousDlqin2yrs"
pastdue=["NumberOfTime30-59DaysPastDueNotWorse","NumberOfTime60-89DaysPastDueNotWorse","NumberOfTimes90DaysLate"]
d = pd.read_csv("../data/raw/cs-training.csv", index_col=0).copy()
d.loc[d["age"]==0,"age"]=np.nan
for c in pastdue: d.loc[d[c].isin([96,98]),c]=np.nan
d["MonthlyIncome_missing"]=d["MonthlyIncome"].isna().astype(int)
Xa=d.drop(columns=[TARGET]); ya=d[TARGET].values
Xtr,Xte,ytr,yte=train_test_split(Xa,ya,test_size=.2,stratify=ya,random_state=42)
Xtr,Xva,ytr,yva=train_test_split(Xtr,ytr,test_size=.25,stratify=ytr,random_state=42)
spw=(ytr==0).sum()/(ytr==1).sum()

med=Xtr.median(); tr_i=Xtr.fillna(med); mean=tr_i.mean(); std=tr_i.std().replace(0,1)
prep=lambda A: torch.tensor(((A.fillna(med)-mean)/std).values, dtype=torch.float32)
Xtr_t,Xva_t,Xte_t=prep(Xtr),prep(Xva),prep(Xte)
ytr_t=torch.tensor(ytr,dtype=torch.float32).unsqueeze(1)
n=Xtr_t.shape[1]

mlp=nn.Sequential(nn.Linear(n,64),nn.ReLU(),nn.Dropout(0.2),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1))
print(f"MLP: {n} -> 64 -> 32 -> 1  ({sum(p.numel() for p in mlp.parameters())} weights)")
criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([spw],dtype=torch.float32))
opt=torch.optim.Adam(mlp.parameters(), lr=1e-3)
loader=DataLoader(TensorDataset(Xtr_t,ytr_t), batch_size=512, shuffle=True)

best_va,best_state=0,None
for epoch in range(1,21):
    mlp.train()
    for xb,yb in loader:
        opt.zero_grad(); loss=criterion(mlp(xb),yb); loss.backward(); opt.step()
    mlp.eval()
    with torch.no_grad():
        va=roc_auc_score(yva, torch.sigmoid(mlp(Xva_t)).numpy().ravel())
    if va>best_va: best_va,best_state=va,{k:v.clone() for k,v in mlp.state_dict().items()}
    if epoch in (1,5,10,15,20): print(f"  epoch {epoch:2d}  val AUC {va:.4f}")
mlp.load_state_dict(best_state); mlp.eval()
with torch.no_grad():
    mlp_auc=roc_auc_score(yte, torch.sigmoid(mlp(Xte_t)).numpy().ravel())
print(f"\nMLP test AUC = {mlp_auc:.4f}")

MLP: 11 -> 64 -> 32 -> 1  (2881 weights)


  epoch  1  val AUC 0.8214


  epoch  5  val AUC 0.8271


  epoch 10  val AUC 0.8296


  epoch 15  val AUC 0.8309


  epoch 20  val AUC 0.8309

MLP test AUC = 0.8366


## 4. Honest finale

```
XGBoost (from notebook 04, same split) : 0.8686
MLP (this notebook)                    : ~0.837
```

**XGBoost wins on static tabular data**, by ~0.03 AUC. The MLP learns real signal but loses, because
trees are built for tabular data (mixed scales, missing values, sharp thresholds, interactions) while
a neural net must learn all that from smooth weighted sums, needs scaling + imputation, and usually
more data. This is the field's consensus, not a quirk of our setup.

## 5. Recap — the point of Phase 3

- A neuron IS logistic regression; a network is neurons stacked with ReLU nonlinearities.
- Learning = forward -> loss -> gradient -> update, repeated; **autograd** computes gradients so we can
  go arbitrarily deep (that gradient-through-layers computation is **backpropagation**).
- **Deep learning does not beat XGBoost on static tables.** It wins where data has structure a tree
  can't see: images, text, and **sequences**.
- That is exactly why the project's headline model is a **hybrid**: XGBoost for static features + an
  **LSTM** for the 12-month payment *sequence* (improving vs deteriorating over time). Next phase.